# N-BEATS: Когда MLP достаточно

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/11_nbeats.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q neuralforecast statsforecast pandas numpy matplotlib

## Подготовка данных

In [ ]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE

# Создаём синтетические данные для демонстрации
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')
y = 100 + np.cumsum(np.random.randn(365)) + 20 * np.sin(np.arange(365) / 7 * 2 * np.pi)

train = pd.DataFrame({
    'unique_id': 'series_1',
    'ds': dates,
    'y': y
})
print(train.head())

## Обучение N-BEATS

In [ ]:
# Параметры
HORIZON = 16
SEASON_LENGTH = 7

# Инициализируем модель
# generic конфигурация — модель сама выбирает базис
model_generic = NBEATS(
    h=HORIZON,                      # горизонт прогноза
    input_size=2 * HORIZON,         # длина входного окна
    loss=MAE(),                     # функция потерь
    max_steps=1000,                 # количество шагов обучения
    stack_types=['generic'] * 2,    # два generic стека
    n_blocks=[1, 1],                # по одному блоку в стеке
    mlp_units=[[512, 512], [512, 512]],  # размеры скрытых слоёв
    scaler_type='standard',         # нормализация входов
    random_seed=42
)

# interpretable конфигурация — фиксированный базис
model_interpretable = NBEATS(
    h=HORIZON,
    input_size=2 * HORIZON,
    loss=MAE(),
    max_steps=1000,
    stack_types=['trend', 'seasonality'],  # тренд + сезонность
    n_blocks=[2, 2],
    n_harmonics=2,                  # количество гармоник для сезонности
    n_polynomials=2,                # степень полинома для тренда
    mlp_units=[[512, 512], [512, 512]],
    scaler_type='standard',
    random_seed=42
)

# Обучаем generic модель
nf = NeuralForecast(
    models=[model_generic],
    freq='D'
)
nf.fit(df=train)

# Прогнозируем
forecasts = nf.predict()

# forecasts содержит колонки: unique_id, ds, NBEATS
print(forecasts.head())

## Оценка качества

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive

# Бейзлайн
sf = StatsForecast(
    models=[SeasonalNaive(season_length=SEASON_LENGTH)],
    freq='D'
)
sf.fit(train)
baseline_forecasts = sf.predict(h=HORIZON)

# Считаем MASE для каждого ряда
def compute_mase(y_true, y_pred, y_train, season_length):
    """
    y_true: фактические значения на тесте
    y_pred: прогноз модели
    y_train: обучающая выборка (для расчёта знаменателя)
    season_length: период сезонности
    """
    numerator = np.mean(np.abs(y_true - y_pred))
    
    # MAE наивного сезонного прогноза на обучении
    naive_errors = np.abs(
        y_train[season_length:] - y_train[:-season_length]
    )
    denominator = np.mean(naive_errors)
    
    return numerator / denominator

print("Прогноз N-BEATS:")
print(forecasts)
print("\nПрогноз SeasonalNaive:")
print(baseline_forecasts)

## Визуализация

In [ ]:
import matplotlib.pyplot as plt

# Выбираем один ряд для визуализации
sample_uid = train['unique_id'].iloc[0]
sample_data = train[train['unique_id'] == sample_uid].tail(100)

fig, ax = plt.subplots(figsize=(12, 5))

# Исходный ряд
ax.plot(sample_data['ds'], sample_data['y'], label='Факт', color='blue')

# Прогноз N-BEATS
forecast_data = forecasts[forecasts['unique_id'] == sample_uid]
ax.plot(forecast_data['ds'], forecast_data['NBEATS'], label='N-BEATS прогноз', color='red', linestyle='--')

ax.set_title('N-BEATS: прогноз временного ряда')
ax.set_xlabel('Дата')
ax.set_ylabel('Значение')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()